[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jordy-Castillo/FC_Jordy-Castillo-Trabajos/blob/main/Taller08/FC_Taller08_JordyCastillo.ipynb)

# **FISICA COMPUTACIONAL - TALLER #8-9**
* **Tema:** Máximos, mínimos y ceros de una función
* **Lenguaje:** Fortran & Python
* **Estudiante:** Jordy Castillo Jordán
* **Fecha:** 12/11/2026



In [104]:
!which gfortran || (apt-get update -qq && apt-get install -y gfortran)
!gfortran --version | head -n 1
!mkdir -p taller_cuerpo_negro

/usr/bin/gfortran
GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0



---
# **PARTE A:**

La radiancia espectral de un cuerpo negro a temperatura $T$ está dada por la **ley de Planck**:

$$B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/(\lambda k_B T)} - 1} ,$$

donde $B_\lambda$ es la potencia emitida por unidad de área, de ángulo sólido y de longitud de onda [W m⁻² sr⁻¹ m⁻¹], $\lambda$ la longitud de onda [m], $T$ la temperatura [K], $h = 6.62607015\times10^{-34}$ J s la constante de Planck, $c = 299792458$ m/s la rapidez de la luz en el vacío y $k_B = 1.380649\times10^{-23}$ J/K la constante de Boltzmann. El factor $2hc^2/\lambda^5$ cuenta los modos electromagnéticos con su energía y el factor con el exponencial es la ocupación de esos modos. Estas tres constantes son exactas en el Sistema Internacional.

**Variable adimensional.** Con $x = hc/(\lambda k_B T)$, la razón entre la energía de un fotón y la energía térmica, el espectro queda

$$B_\lambda \propto \frac{x^5}{e^x - 1} .$$

**Ley de desplazamiento de Wien.** El máximo del espectro cumple $dB_\lambda/dx = 0$, que se reduce a

$$x = n\left(1 - e^{-x}\right), \qquad n = 5 ,$$

ecuación trascendente sin solución en funciones elementales, con la raíz trivial $x = 0$ y una raíz positiva $x_5$. Esa raíz fija la constante de Wien $b$:

$$\lambda_{\max}\,T = b = \frac{hc}{k_B\,x_5} .$$

Si el espectro se expresa por unidad de frecuencia, $B_\nu \propto x^3/(e^x - 1)$, y el máximo cumple la misma ecuación con $n = 3$.

**Ley de Stefan-Boltzmann.** La potencia total emitida por unidad de área es $M = \int_0^\infty \pi B_\lambda\,d\lambda = \sigma T^4$, con

$$\sigma = \frac{2\pi k_B^4}{h^3 c^2}\,I, \qquad I = \int_0^\infty \frac{x^3}{e^x - 1}\,dx ,$$

donde $\sigma$ es la constante de Stefan-Boltzmann [W m⁻² K⁻⁴] e $I$ una integral pura, cuyo valor exacto es $\pi^4/15$. Es una integral impropia. El cambio de variable $x = t/(1 - t)$, con $dx = dt/(1-t)^2$, la lleva al intervalo finito $[0, 1]$:

$$I = \int_0^1 \frac{x(t)^3}{e^{x(t)} - 1}\,\frac{dt}{(1 - t)^2} .$$

**El taller tiene tres partes:**

| Parte | Lenguaje | Qué se hace |
|---|---|---|
| B | Fortran | Módulo con tres métodos para hallar ceros; constante de Wien |
| C | Fortran | Módulo de Romberg del Taller 7 con un integrando nuevo; constante de Stefan-Boltzmann |
| D | Python | Máximo del espectro solar en una malla, con spline, comparado con la ley de Wien |

**Requisitos generales**

1. Todo el trabajo, incluido este notebook, se guarda en una carpeta llamada `taller_cuerpo_negro`, con los nombres de archivo que se indican en cada parte. El archivo `modulo_romberg.f90` del Taller 7 se copia en esa carpeta.
2. Se usa doble precisión: `real(dp)` en Fortran y `float` (64 bits) en Python.
3. Los métodos para hallar ceros tienen un máximo de 100 iteraciones y terminan con un mensaje si no convergen. Romberg conserva el límite de niveles del Taller 7.
4. La bisección comprueba el cambio de signo antes de iterar. Newton y secante se detienen con un mensaje si el denominador es nulo.
5. Se compila con `-fcheck=bounds`.
6. Los números reales de los archivos `.dat` se escriben con formato `es24.16`, y los identificadores y conteos como enteros.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `scipy.optimize`, `scipy.integrate`, `np.gradient`, `np.trapz` ni funciones de biblioteca que hallen ceros, integren o deriven. `CubicSpline`, con sus métodos `.roots()` y `.derivative()`, se permite solo en la Parte D; las diferencias centrales se programan. Los métodos se programan a partir de las secciones 2 a 5. Cada estudiante sustentará su programa.

</div>


---
# **PARTE B:**


1. Escriba `modulo_ceros.f90` con la función $F(x) = x - n(1 - e^{-x})$, su derivada y los métodos de bisección, Newton y secante. La organización en procedimientos es decisión suya; $n$ se pasa como argumento y debe respetar las reglas de la Semana 6.
2. Escriba `programa_wien.f90`, que use ese módulo para hallar $x_5$ y $x_3$ con tolerancia $\varepsilon = 10^{-10}$, con estos datos iniciales fijos:

| Método | Datos iniciales | Parada |
|---|---|---|
| Bisección | $[a, b] = [1, 10]$ | $b - c \le \varepsilon$ (sección 2) |
| Newton | $x_0 = 10$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |
| Secante | $x_0 = 10$, $x_1 = 9$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |

3. Repita Newton para $n = 5$ con $x_0 = 0.5$, y explique el resultado en el reporte.
4. Calcule la constante de Wien $b$ con $x_5$ obtenido por Newton.
5. Registre cada iteración de los tres métodos con $n = 5$ para analizar la convergencia. Las iteraciones de cada método se numeran desde 1 y en orden: la primera es la que produce el primer valor nuevo ($c_1$ en bisección, $x_1$ en Newton, $x_2$ en la secante).

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_ceros.f90` | Función, derivada y los tres métodos, con conteo de iteraciones |
| `programa_wien.f90` | Usa el módulo; escribe los tres archivos `.dat` de esta parte |
| `resultado_wien.dat` | 7 filas: $n$, método, raíz, iteraciones. Método: 1 bisección, 2 Newton, 3 secante, para $n = 5$ y $n = 3$; la fila 7 es $n = 5$, método 4 (Newton con $x_0 = 0.5$) |
| `iteraciones_wien.dat` | Una fila por iteración con $n = 5$: método (1 a 3), número de iteración, valor de $x$ |
| `constante_wien.dat` | Una línea: $x_5$, $x_3$, $b$ [m K] |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** La bisección para $n = 5$ debe usar el número de iteraciones que predice la fórmula de la sección 2. Calcúlelo antes de ejecutar el programa: si no coincide, revise el criterio de parada.

</div>


### **B.1** Modulo_ceros.f90


In [130]:
#ELABORACION MEDIANTE MERMAID
from IPython.display import Image
url_imagen = "https://github.com/Jordy-Castillo/FC_Jordy-Castillo-Trabajos/blob/main/Taller08-09/T8_DiagramaB1.png?raw=true"
Image(url=url_imagen, width=1100)

In [106]:
%%writefile taller_cuerpo_negro/modulo_ceros.f90
!!!!!!!!! DEFINIR LA FUNCION Y SUS METODOS !!!!!!!!!

module modulo_ceros
  implicit none
  integer, parameter :: dp = kind(1.0d0)                                        !definir precisión

!! FUNCION ORIGINAL !!
contains
  real(dp) function F(x, n)
    real(dp), intent(in) :: x                                                   !raíz de la función
    real(dp), intent(in) :: n                                                   !número de la función
    F = x - n * (1.0_dp - exp(-x))                                              !definimos la funcion que busca el cero
  end function F

!! FUNCION DERIVADA !!

  real(dp) function DF_F(x,n)
    real(dp), intent(in) :: x                                                   !raíz de la función
    real(dp), intent(in) :: n                                                   !número de la función
    DF_F = 1.0_dp - n * exp(-x)                                                 !definimos la derivada de la funcion (calculada a mano)
  end function DF_F

!! METODO BISECCION !!

  subroutine biseccion(n, a_int, b_int, tol, max_inter, raiz, iteracciones, u_hist)
    real(dp), intent(in)  :: n, a_int, b_int, tol                  !definir los parametros de entrada
    integer, intent(in)   :: max_inter                             !definir el maximo de iteraciones
    real(dp), intent(out) :: raiz                                  !definir la raiz que usaremos
    integer, intent(out)  :: iteracciones                          !definir el numero de iteraciones
    integer, intent(in), optional :: u_hist                        !definimos el "historial" de iteracciones con "Optional" para que no sea obligatorio
    real(dp) :: a, b, c                                            !variables para proteger los datos originales
    integer :: i                                                   !contador arbitrario

    a = a_int                                                                   !#re-definimos los datos de entrada para poder modificarlos
    b = b_int                                                                   !#recordando que las variables de lectura no se pueden modificar
    iteracciones = 0

    if (F(a, n) * F(b, n) >= 0.0_dp) then                                       !verificar el cambio del signo incial
       stop "Bisección: f(a) y f(b) no tienen signos distintos"
    end if

    do i = 1, max_inter                                                         !punto medio del intervalo
      c = 0.5_dp * (a + b)                                                      !conteo de las iteracciones
      iteracciones = i

      if (present(u_hist)) then                                                 !##"present" indetifica si u_hist fue dado o no por el programa
         write(u_hist, '(i2, i4, es24.16)') 1, i, c                             !##lo hacemos para resutilizar el codigo mas adelante
      end if

      if ((b - c) <= tol) then                                                  !criterio de parada
        raiz = c                                                                !guardar la raiz
        return                                                                  !terminar la subrutina
      end if

      if (F(a,n) * F(c,n) <= 0.0_dp) then                                       !actualizar el intervalo de la cota de error
        b = c
      else
        a = c
      end if
    end do

    raiz = c                                                                    !si se llega aqui, no se cumplio el criterio de parada
    print *, "Bisección: No convergió en max_inter iteraciones"
  end subroutine biseccion


!! METODO NEWTON-RAPHSON !!

  subroutine newton(n, x_int, tol, max_inter, raiz, iteracciones, u_hist)
    real(dp), intent(in)  :: n, x_int, tol                          !definir los parametros de entrada
    integer, intent(in)   :: max_inter                              !definir el maximo de iteraciones
    real(dp), intent(out) :: raiz                                   !definir la raiz que usaremos
    integer, intent(out)  :: iteracciones                           !definir el numero de iteracciones
    integer, intent(in), optional :: u_hist                         !definimos el "historial" de iteracciones con "Optional" para que no sea obligatorio
    real(dp) :: x_cu, x_nex, paso, dfx                              !variables para proteger los datos originales
    integer :: i                                                    !contador arbitrario

    x_cu = x_int                                                    !#re-definimos los datos de entrada para poder modificarlos
    x_nex = x_cu                                                    !#recordando que las variables de lectura no se pueden modificar
    iteracciones = 0

    do i = 1, max_inter                                             !contador arbitrario comienzo con ciclo do
      dfx = DF_F(x_cu,n)                                            !definimos la funcion derivada
      if (abs(dfx) < tiny(1.0_dp)) then
         stop "Newton-Raphson: Derivada Nula o muy cercana a cero"
      end if

      paso = -f(x_cu,n) / dfx                                        !calculo del paso con correcion de Newton
      x_nex = x_cu + paso                                            !calculo de la nueva aproximacion
      iteracciones = i                                               !conteo de las iteracciones

      if (present(u_hist)) then                                                 !##"present" indetifica si u_hist fue dado o no por el programa
         write(u_hist, '(i2, i4, es24.16)') 2, i, x_nex
      end if

      if (abs(paso) < tol) then                                                 !criterio de parada por paso
         raiz = x_nex                                                           !actualizar la aproximacion
         return
      end if
      x_cu = x_nex
    end do

    raiz = x_nex
    print *, "Newton-Raphson; No convergió en max_inter iteraciones"
  end subroutine newton


!! METODO DE LA SECANTE !!

  subroutine secante(n, x_int, x_int_1, tol, max_inter, raiz, iteracciones, u_hist)
    real(dp), intent(in)  :: n, x_int, x_int_1, tol                !definir los parametros de entrada
    integer, intent(in)   :: max_inter                             !definir el maximo de  iteraciones
    real(dp), intent(out) :: raiz                                  !definir la raiz que usaremos
    integer, intent(out)  :: iteracciones                          !definir el numero de iteracciones
    integer, intent(in), optional :: u_hist
    real(dp) :: x_cu, x_nex, paso                                  !variables para proteger los datos originales
    real(dp) :: x_aux, f_aux, f_cu, denom                          !variables para proteger los datos originales
    integer :: i                                                   !contador arbitrario

    x_aux = x_int                                                  !#re-definimos los datos de entrada para poder modificarlos
    x_cu  = x_int_1                                                !#recordando que las variables de lectura no se pueden modificar
    x_nex = x_cu                                                   !inicializar x_nex para que nunca quede sin valor
    f_aux = F(x_aux, n)                                            !inicializar f_aux para que nunca quede sin valor
    iteracciones = 0                                               !inicar contador en 0

    do i = 1, max_inter                                            !contador arbitrario
      f_cu  = F(x_cu, n)                                           !calcular las funciones en los puntos de f_cu
      denom = f_cu - f_aux                                         !calculo del denominador

      if (abs(denom) < tiny(1.0_dp)) then
         stop "Secante: Dominador Nulo o muy cercano a cero"
      end if

      paso = -f_cu * (x_cu - x_aux) / denom
      x_nex = x_cu + paso
      iteracciones = i                                                   !En la secante, el primer valor nuevo corresponde a x_2

      if (present(u_hist)) then
         write(u_hist, '(i2, i4, es24.16)') 3, i, x_nex
      end if

      if (abs(paso) < tol) then                                                 !criterio de parada por paso
         raiz = x_nex
         return
      end if

      x_aux = x_cu                                                              !actualizar los puntos de la secante
      f_aux = f_cu
      x_cu  = x_nex
    end do

    raiz = x_nex
    print *, "Secante: No convergió en max_inter iteraciones"
  end subroutine secante

end module modulo_ceros

Overwriting taller_cuerpo_negro/modulo_ceros.f90



### **B.2-B.3-B.4-B.5** Programa_wien.f90


In [131]:
#ELABORACION MEDIANTE MERMAID
from IPython.display import Image
url_imagen = "https://github.com/Jordy-Castillo/FC_Jordy-Castillo-Trabajos/blob/main/Taller08-09/T8_DiagramaB2.png?raw=true"
Image(url=url_imagen, width=1100)

In [108]:
%%writefile taller_cuerpo_negro/programa_wien.f90
!!!!!!!!!! ESCRIBIR LOS 3 ARCHIVOS DE SALIDA, LA CONSTANTE WEIN Y X5 Y X3 !!!!!!!!!!

program programa_wien
    use modulo_ceros, only: dp, biseccion, newton, secante
    implicit none

    real(dp), parameter :: tol = 1.0e-10_dp                                            !definimos la tolerancia epsilon dada
    integer, parameter  :: max_iter = 100                                              !maximo de iteraciones
    real(dp), parameter :: h_val  = 6.62607015e-34_dp                                  !constante de Planck [J*s]
    real(dp), parameter :: c_val  = 299792458.0_dp                                     !velocidad de la luz [m/s]
    real(dp), parameter :: kb_val = 1.380649e-23_dp                                    !constante de Boltzmann [J/K]

    real(dp) :: x5, x3, b_wien                                                         !definimos las variables que usaremos
    real(dp) :: raiz_res                                                               !raiz resultante
    integer :: it_res, u_res, u_it, u_cte                                              !variables para los archivos de salida

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! GENERAR ARCHIVO RESULTADO.WIEN.DAT E ITERACIONES_WIEN.DAT !!!

!! Las iteraciones de n=5 las registra el propio modulo (u_hist), asi la raiz salen del mismo codigo !!

    open(newunit=u_res, file='resultado_wien.dat',   status='replace', action='write')      !archivo de resultados
    open(newunit=u_it,  file='iteraciones_wien.dat', status='replace', action='write')      !archivo de iteraciones

    call biseccion(5.0_dp, 1.0_dp, 10.0_dp, tol, max_iter, raiz_res, it_res, u_hist=u_it)   !biseccion n=5 (registra iteraciones)
    write(u_res, '(i2, i2, es24.16, i4)') 5, 1, raiz_res, it_res                            !se escribe en el archivo de salida

    call newton(5.0_dp, 10.0_dp, tol, max_iter, x5, it_res, u_hist=u_it)                    !Newton n=5 (registra iteraciones)
    write(u_res, '(i2, i2, es24.16, i4)') 5, 2, x5, it_res                                  !se escribe en el archivo de salida

    call secante(5.0_dp, 10.0_dp, 9.0_dp, tol, max_iter, raiz_res, it_res, u_hist=u_it)     !secante n=5 (registra iteraciones)
    write(u_res, '(i2, i2, es24.16, i4)') 5, 3, raiz_res, it_res                            !se escribe en el archivo de salida

    call biseccion(3.0_dp, 1.0_dp, 10.0_dp, tol, max_iter, raiz_res, it_res)                !se ejecuta la bisección para n=3
    write(u_res, '(i2, i2, es24.16, i4)') 3, 1, raiz_res, it_res                            !se escribe en el archivo de salida

    call newton(3.0_dp, 10.0_dp, tol, max_iter, x3, it_res)                                 !se ejecuta el Newton-Raphson para n=3
    write(u_res, '(i2, i2, es24.16, i4)') 3, 2, x3, it_res                                  !se escribe en el archivo de salida

    call secante(3.0_dp, 10.0_dp, 9.0_dp, tol, max_iter, raiz_res, it_res)                  !se ejecuta la secante para n=3
    write(u_res, '(i2, i2, es24.16, i4)') 3, 3, raiz_res, it_res                            !se escribe en el archivo de salida

    !!!! PUNTO B.3 !!!!
    call newton(5.0_dp, 0.5_dp, tol, max_iter, raiz_res, it_res)                            !se ejecuta el Newton-Raphson para n=5 con x0 = 0.5
    write(u_res, '(i2, i2, es24.16, i4)') 5, 4, raiz_res, it_res                            !se escribe en el archivo de salida

    close(u_res)
    close(u_it)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! GENERAR ARCHIVIO CONSTANTE_WIEN.DAT !!!
    b_wien = (h_val * c_val) / (kb_val * x5)                                                !se calcula la constante de Wien mediante su ecuacion

    open(newunit=u_cte, file='constante_wien.dat', status='replace', action='write')
    write(u_cte, '(3es24.16)') x5, x3, b_wien
    close(u_cte)

    print *, "La parte B ha sido ejecutada de forma correcta"
end program programa_wien

Overwriting taller_cuerpo_negro/programa_wien.f90


In [109]:
#Compilar y ejecutar la Parte B
!cd taller_cuerpo_negro && gfortran -fcheck=bounds -o programa_wien modulo_ceros.f90 programa_wien.f90 && ./programa_wien

 La parte B ha sido ejecutada de forma correcta



---
# **PARTE C:**

1. Agregue a su `modulo_romberg.f90` del Taller 7 los integrandos de esta parte, sin cambiar la integración ni su criterio de parada. Los extremos $t = 0$ y $t = 1$ deben dar el límite correcto del integrando, que es cero en ambos.
2. Escriba `programa_sigma.f90`, que calcule $I$ en $t \in [0, 1]$ con tolerancia $10^{-8}$ y obtenga $\sigma$.
3. Calcule también $\int_0^{100} x^3/(e^x - 1)\,dx$ con el mismo Romberg y la misma tolerancia, **sin** cambio de variable.
4. Compare los dos resultados con $\pi^4/15$ y explique en el reporte lo que ocurre en el paso 3.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_romberg.f90` | Módulo del Taller 7 con los integrandos nuevos |
| `programa_sigma.f90` | Usa el módulo; escribe el archivo `.dat` de esta parte |
| `resultado_sigma.dat` | Dos filas, $k_{\text{final}}$, evaluaciones, integral, $\sigma$: fila 1 con cambio de variable, fila 2 sin cambio de variable. El nivel $k = 0$ es el trapecio inicial y las evaluaciones son $2^k + 1$ |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Para $t$ cercano a 1, $x$ es tan grande que $e^x$ desborda la doble precisión a partir de $x \approx 709$. El integrando es prácticamente cero mucho antes: decida a partir de qué valor de $x$ puede devolver cero.

</div>


### **C.1** Actualizacion del modulo_romberg


In [110]:
%%writefile taller_cuerpo_negro/modulo_romberg.f90
!!!!!!!!!! AÑADIR NUEVOS INTEGRANDOS AL MODULO !!!!!!!!!!

module modulo_romberg
  implicit none
  integer, parameter :: dp = kind(1.0d0)                          !definir precisión

!! INTEGRAMOS LA POTENCIA DISIPADA POR FRICCION !!

contains
  real(dp) function f_energia(t)                                  ! Funcion integrando 2gamma*v(t)^2
    real(dp), intent(in) :: t                                     ! Tiempo (s)
    real(dp), parameter  :: gamma = 0.5_dp                        ! Coeficiente de amortiguamiento (s^-1)
    real(dp), parameter  :: w = 2.0_dp *4.0_dp* atan(1.0_dp)      ! Frecuencia angular de oscilación (rad/s) con PI = 4*arctan(1)
    real(dp) :: v                                                 ! Velocidad (m/s)

    v = -exp(-gamma*t) * (gamma*cos(w*t) + w*sin(w*t))            !Calculo de la Velocidad en funcion del tiempo v(t)
    f_energia = 2.0_dp * gamma * v**2                             !Calculo de la energia
  end function f_energia

!! INTEGRAMOS LA FUNCION PARA EL CALCULO PI > 4/(1+X^2) !!

  real(dp) function f_pi(x)                                       ! Funcion integrando 4/(1+x^2)
    real(dp), intent(in) :: x                                     ! Posicion
    f_pi = 4.0_dp / (1.0_dp + x**2)                               !Calculo de la funcion
  end function f_pi

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

!! INTEGRADORES NUEVOS PARA LA PARTE C (NUEVA PARTE) !!

  !!!! CON CAMBIO DE VARIABLE EN [0,1] !!!!
  real(dp) function f_sigma_t(t)                                  !x(t) = t / (1 - t), dx = dt / (1 - t)^2
    real(dp), intent(in) :: t
    real(dp) :: x, exp_val                                        !Varaibles

    if (t <= 0.0_dp .or. t >= 1.0_dp) then                        !controlar extremos para evitar indeterminaciones
       f_sigma_t = 0.0_dp                                         !no hay cambio de variable
       return
    end if

    x = t / (1.0_dp - t)                                          !definir la variable para su cambio de variable

    if (x > 700.0_dp) then                                        !proteger ante un desbordamiento si (x) es muy grande
       f_sigma_t = 0.0_dp                                         !no hay cambio de variable
       return
    end if

    exp_val = exp(x)                                              !re-definimos la exponencial
    f_sigma_t = (x**3 / (exp_val - 1.0_dp)) / ((1.0_dp - t)**2)   !definimos la funcion original (I)
  end function f_sigma_t

  !!!! SIN CAMBIO DE VARIABLE EN [0,100] !!!!
  real(dp) function f_sigma_x(x)                                  !f(x) = x^3 / (e^x - 1)
    real(dp), intent(in) :: x

    if (x <= 0.0_dp) then                                         !controlar extremos para evitar indeterminaciones
       f_sigma_x = 0.0_dp                                         !no hay cambio de variable
       return
    end if

    if (x > 700.0_dp) then                                        !proteger ante un desbordamiento si (x) es muy grande
       f_sigma_x = 0.0_dp                                         !no hay cambio de variable
       return
    end if

    f_sigma_x = (x**3) / (exp(x) - 1.0_dp)                        !definimos la funcion original
  end function f_sigma_x


  !!!! SELECTOR DE INTEGRANDOS PARA ESTE TALLER !!!!
  real(dp) function eval_f(x, tipo_funcion)                       !creamos una funcion evaluadora
    real(dp), intent(in) :: x                                     !variable
    integer, intent(in) :: tipo_funcion                           !tipo de funcion

    select case (tipo_funcion)
    case (1)
       eval_f = f_energia(x)                                      !funcion de energia con cambio de variable [0, 1]
    case (2)
       eval_f = f_pi(x)                                           !funcion de romberg sin cambio de variable [0, 100]
    case (3)
       eval_f = f_sigma_t(x)                                      !funcion con cambio de variable [0, 1]
    case (4)
       eval_f = f_sigma_x(x)                                      !funcion sin cambio de variable [0, 100]
    case default
       eval_f = 0.0_dp                                            !funcion hay indeterminacion o no es valida
    end select
  end function eval_f

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

!! SUBRUTINA PARA LA INTEGRACION DE ROMBERG !!

  subroutine integrar_romberg(a, b, tol, kmax, tipo_funcion, S, k_final, n_eval)
    real(dp), intent(in)  :: a, b, tol                            !Rango de integracion (a y b), tolerancia
    integer, intent(in)   :: kmax, tipo_funcion                    !Numero de niveles y tipo de funcion
    real(dp), intent(out) :: s(0:, 0:)                            !Arreglo de resultados
    integer, intent(out)  :: k_final, n_eval                      !Nivel final y numero de evaluaciones
    integer  :: k, j, i, puntos_nuevos                             !Variables para el bucle
    real(dp) :: h, suma, x_nuevo                                  !Variables para el bucle

    !INICIAR EL NIVEL K = 0
    h = b - a
    s(0,0) = 0.5_dp * h * (eval_f(a, tipo_funcion) + eval_f(b, tipo_funcion))    !regla del trapecio cuando K=0
    n_eval = 2                                                                   !inicia un contador en 2 evaluaciones (f(a) y f(b))
    k_final = kmax                                                               !inicia el nivel maximo en el nivel maximo

    !BUCLE PARA EL NIVEL K = 1, 2, ..., KMAX
    do k = 1, kmax                                                               !realiza los refinamientos de la malla
      h = h/2.0_dp                                                               !se reduce el ancho del intervalo
      suma = 0.0_dp                                                              !se reinicia la suma
      puntos_nuevos = 2**(k-1)                                                   !se calcula el numero de puntos nuevos en el nivel k

      do i = 1, puntos_nuevos
        x_nuevo = a + real(2*i-1, dp) * h                                        !puntos medios impares del intervalo
        suma = suma + eval_f(x_nuevo, tipo_funcion)                              !se suma la funcion en los puntos medios impares
      end do

      n_eval = n_eval + puntos_nuevos                                            !incrementa el contador de evaluaciones
      S(k,0) = 0.5_dp * S(k-1,0) + h * suma                                      !aplicar formula de refinamiento del trapecio

      do j = 1, k
        S(k,j) = ((4.0_dp**j) * S(k, j-1) - S(k-1, j-1)) / ((4.0_dp**j) - 1.0_dp)  !extrapolacion de richardson para las columnas superiores
      end do

      if (abs(S(k,k) - S(k-1, k-1)) < tol ) then                                 !criterio de parada
        k_final = k                                                              !se actualiza el nivel final
        exit                                                                     !salir del bucle
      end if
    end do
  end subroutine integrar_romberg
end module modulo_romberg

Overwriting taller_cuerpo_negro/modulo_romberg.f90



### **C.2** Programa sigma.f90


In [111]:
%%writefile taller_cuerpo_negro/programa_sigma.f90
!!!!!!!!!! USAR MODULO Y CALCULAR (I) EN t[0,1], Y OBTENER σ !!!!!!!!!!

program programa_sigma
    use modulo_romberg
    implicit none

    real(dp), parameter :: tol = 1.0e-8_dp                                      !definir la tolerancia pedida
    integer, parameter  :: max_k = 25                                           !definir el numero maximo de niveles
    real(dp), parameter :: h_val  = 6.62607015e-34_dp                           !constante de Planck [J*s]
    real(dp), parameter :: c_val  = 299792458.0_dp                              !velocidad de la luz [m/s]
    real(dp), parameter :: kb_val = 1.380649e-23_dp                             !constante de Boltzmann [J/K]

    real(dp) :: integral_t, integral_x, sigma_t, sigma_x, factor_pre            !definir las variables que usaremos
    real(dp), allocatable :: S(:,:)                                             !Arreglo de maximo de niveles
    integer :: k_fin_t, neval_t, k_fin_x, neval_x, u_out                        !definir las variables para los archivos de salida

    allocate(S(0:max_k, 0:max_k))                                               !Reservar memoria para el arreglo

    !! Factor multiplicativo para obtener sigma a partir de I !!
    factor_pre = (2.0_dp * acos(-1.0_dp) * kb_val**4) / (h_val**3 * c_val**2)    !definicion del factor de precalculo para no saturar el codigo despues


    !! Calculo con cambio de variable en t in [0, 1] !!
    call integrar_romberg(0.0_dp, 1.0_dp, tol, max_k, 3, S, k_fin_t, neval_t)          !calcular la integral con cambio de variable
    integral_t = S(k_fin_t, K_fin_t)                                                   !obtener la integral final (en t)
    sigma_t = factor_pre * integral_t                                                !obtener el valor de sigma con la ecuación


    !! Calculo sin cambio de variable en x in [0, 100] !!
    call integrar_romberg(0.0_dp, 100.0_dp, tol, max_k, 4, S, k_fin_x, neval_x)        !calcular la integral sin cambio de variable
    integral_x = S(k_fin_x, K_fin_x)                                                   !obtener la integral final (en x)
    sigma_x = factor_pre * integral_x                                                  !obtener el valor de sigma con la ecuación

    !! Escribimos el archivo resultado.sigma.dat !!
    open(newunit=u_out, file='resultado_sigma.dat', status='replace', action='write')

    write(u_out, '(i4, i10, 2es24.16)') k_fin_t, neval_t, integral_t, sigma_t                !fila 1 (cambio de varaible)
    write(u_out, '(i4, i10, 2es24.16)') k_fin_x, neval_x, integral_x, sigma_x                !fila 2 (sin cambio de variable)

    close(u_out)
    deallocate(S)                                                                            !Liberar memoria

    print *, "La parte C fue ejecutada de forma correcta"
end program programa_sigma


Overwriting taller_cuerpo_negro/programa_sigma.f90


In [112]:
#Compilar y ejecutar la Parte C
!cd taller_cuerpo_negro && gfortran -fcheck=bounds -o programa_sigma modulo_romberg.f90 programa_sigma.f90 && ./programa_sigma

 La parte C fue ejecutada de forma correcta



---
# **PARTE D:**

Se modela el Sol como un cuerpo negro a $T = 5772$ K. El espectro se muestrea en una malla uniforme de longitudes de onda en $[100, 3000]$ nm, con paso $\Delta\lambda$.

1. Escriba `maximo_espectro.py`. Construya $B_\lambda$ para $\Delta\lambda = 100, 50, 25$ y $12.5$ nm.
2. Para cada malla, obtenga $\lambda_{\max}$ de tres maneras: (a) mayor valor del arreglo; (b) cero del spline de la derivada central; (c) cero de la derivada del spline de $B_\lambda$. En (b) y (c), conserve los ceros donde la segunda derivada es negativa y, si queda más de uno, el más cercano al resultado de (a): en una malla gruesa el spline oscila y puede producir ceros espurios.
3. Lea $b$ de `constante_wien.dat` y obtenga el error absoluto en nm de cada resultado frente a $\lambda_{\max} = b/T$.
4. Escriba `maximo_espectro.dat` y grafique en `espectro_solar.png` el espectro de la malla de 25 nm, con $\lambda_{\max}$ marcado. Agregue en la misma figura el $\lambda$ que corresponde al máximo en frecuencia, $\lambda = hc/(k_B T x_3)$.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `maximo_espectro.py` | Construye los datos, halla los máximos, escribe los archivos y la figura |
| `maximo_espectro.dat` | Cuatro filas, una por $\Delta\lambda$: $\Delta\lambda$ [nm], error (a), error (b), error (c) [nm] |
| `espectro_solar.png` | Espectro con $\lambda_{\max}$ y el $\lambda$ del máximo en frecuencia |

In [113]:
%%writefile taller_cuerpo_negro/maximo_espectro.py

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

#### DEFINICION DE CONSTANTES ####

h_planck = 6.62607015e-34   #Constante de Planck    [J*s]
c_luz = 299792458.0         #Velocidad de la luz    [m/s]
k_b = 1.380649e-23          #Constante de Boltzmann [J/K]
T_sol = 5772.0              #Temperatura del Sol     [K]


#### LECTURA DE CONSTANTE_WIEN.DAT PARA VALOR TEORICO ####

data_wien  = np.loadtxt("constante_wien.dat")                                   #leemos el archivo generado
x3_teorica = data_wien[1]                                                       #obtenemos x3
b_wien     = data_wien[2]                                                       #obtenemos b

lam_max_teor_nm = (b_wien / T_sol) * 1e9                                        #valor teorico del pico espectral solar en nm

#### FUNCION DE PLANCK EN FUNCION DE LA LONGITUD ####

def planck_lambda(lam_nm, T):
    "Calcula la radiancia espectral de un cuerpo negro en W/(m^2 sr m)."
    lamn_m = lam_nm * 1e-9                                                      #convetir nm a metros
    exponente = (h_planck * c_luz) / (lamn_m * k_b * T)                         #definimos lo que esta en el exponente
    factor = (2.0 * h_planck * c_luz**2) / (lamn_m**5)                          #defininos lo que multiplica la ecuacion
    return factor / (np.exp(exponente) - 1.0)                                   #realizamos la ecuacion de la ley de planck

#### EVALUAR LAS 4 MALLAS Y GENERAR EL MAXIMO_ESPECTRO.DAT

pasos_dlam = [100.0, 50.0, 25.0, 12.5]                                          #definimos los pesos de cada malla
resultados_mallas = []                                                          #una lista vacia para guardar los datos

for dlam in pasos_dlam:
  lam = np.arange(100.0, 3000.0 + dlam / 2.0, dlam)                             #construimos la malla de longitud de onda [100 a 3000] nm
  B = planck_lambda(lam, T_sol)                                                 #calculamos el espectro

  ## MANERA A: CON NP.ARGMAX ##
  idx_max = np.argmax(B)                                                        #obtenemos el indice del maximo
  lam_a   = lam[idx_max]                                                        #calculo de lam maximo por manera A
  err_a = abs(lam_a - lam_max_teor_nm)                                          #calculamos el error

  ## MANERA B: CON CERO DEL SPLINE DE LA DERIVADA CENTRAL ##
  d_central = (B[2:] - B[:-2]) / (2.0*dlam)                                     #definicion de la derivada central
  lam_i = lam[1:-1]                                                             #puntos interiores donde opera la derivada central
  s_d = CubicSpline(lam_i, d_central)                                           #spline de la derivada central

  raices_b = s_d.roots(extrapolate=False)                                       #obtenemos los ceros del spline
  raices_b = raices_b[s_d(raices_b, 1) < 0]                                     #filtramos los ceros con segunda derivada negativa

  if len(raices_b) > 0:                                                         #si el numero de raices_b es mayor a 0
     lam_b = raices_b[np.argmin(abs(raices_b - lam_a))]
  else:
     lam_b = lam_a                                                               #si es menor, entonces seran iguales
  err_b   = abs(lam_b - lam_max_teor_nm)                                        #calculamos el error

  ## MANERA C: CON CERO DEL SPLINE EN LA DERIVADA DE B_LAMBDA ##
  s_func = CubicSpline(lam, B)                                                  #spline de B_lambda
  ds_deriv = s_func.derivative()                                                #spline de la derivada de B_lambda

  raices_c = ds_deriv.roots(extrapolate=False)                                  #obtenemos los ceros del spline de la derivada
  validos_c = []                                                                #filtramos los ceros de la segunda derivada negativa
  for r in raices_c:
    if r > 100.0 and ds_deriv(r,1) < 0 :
       validos_c.append(r)                                                       #agregar a la lista de validos

  if len(validos_c) > 0:
    lam_c = min(validos_c, key=lambda val: abs(val - lam_a))
  else:
    lam_c = lam_a                                                               #si es menor, entonces seran iguales
  err_c = abs(lam_c - lam_max_teor_nm)                                          #calculamos el error

  resultados_mallas.append([dlam, err_a, err_b, err_c])                         #guardar fila para maximo_espectro.dat

np.savetxt("maximo_espectro.dat", resultados_mallas, fmt= "%24.16e")            #guardar el .dat

#### GENERAMOS LA GRAFICA ESPECTRO_SOLAR.PNG ####

dlam_g = 25.0                                                                   #con 25nm como pide el D.4
lam_g  = np.arange(100.0, 3000.0 + dlam_g / 2.0, dlam_g)                        #construimos la malla de longitud de onda [100 a 3000] nm
B_g    = planck_lambda(lam_g, T_sol)                                            #calculamos el espectro
lam_plot = np.linspace(100.0, 3000.0, 500)                                      #construimos la malla para la grafica
B_plot = planck_lambda(lam_plot, T_sol)                                         #calculamos el espectro para la linea continua

## Maximo espectral en la longitud de onda ##
lam_max_lambda = lam_max_teor_nm
B_max_lambda   = planck_lambda(lam_max_lambda, T_sol)

## Maximo en frecuencia en la longitud de onda ##
lam_max_nu = ((h_planck * c_luz) / (k_b * T_sol * x3_teorica)) * 1e9
B_max_nu   = planck_lambda(lam_max_nu, T_sol)


## parametros fundamentales para la grafica ##
plt.figure(figsize=(9, 5.5), dpi=250)
plt.plot(lam_plot, B_plot / 1e13, color="orange", lw=2, label="Espectro Solar ($T=5772$ K)")
plt.plot(lam_g, B_g / 1e13, "o", color="purple", ms=4, label=rf"Malla $\Delta\lambda = {dlam_g}$ nm")

plt.plot(lam_max_lambda, B_max_lambda / 1e13, 'bo', markersize=8, label="Máx. en Longitud de Onda")
plt.plot(lam_max_nu, B_max_nu / 1e13, 'go', markersize=8, label="Máx. en Frecuencia")

plt.axvline(lam_max_lambda, color="blue", ls="--", label=rf"MAX $\lambda \approx {lam_max_lambda:.1f}$ nm")
plt.axvline(lam_max_nu, color="green", ls=":", label=rf"MAX $\nu \approx {lam_max_nu:.1f}$ nm")

plt.title("Espectro de Emision del Cuerpo Negro del Sol", fontsize = 16)
plt.xlabel(r"Longitud de onda $\lambda$ [nm]")
plt.ylabel(r"Radiancia Espectral $B_\lambda(\lambda, T)$ [$10^{13}$ W m$^{-2}$ sr$^{-1}$ m$^{-1}$]")
plt.xlim(100,3000)
plt.ylim(bottom=0)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True, facecolor="white", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig("espectro_solar.png", dpi=250)
plt.close()

print("La Parte D ha sido ejecutada correctamente")

Overwriting taller_cuerpo_negro/maximo_espectro.py


In [114]:
#Ejecutar la Parte D
!cd taller_cuerpo_negro && python maximo_espectro.py

La Parte D ha sido ejecutada correctamente



---
# **PARTE E:**

1. **Costo de los métodos.** Compare las iteraciones de los tres métodos con $n = 5$. Con `iteraciones_wien.dat`, estime el orden de Newton y de la secante.

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
Para $n=5$, los tres métodos presentan diferencias en el número de iteraciones y el costo computacional para aproximar la raíz de la ecuación de Wien.

El metodo de bisección requiere $37$ iteraciones. Su convergencia es lineal ($p=1$), por lo que reduce el intervalo a la mitad en cada iteración. En el archivo `iteraciones_wien.dat` aparece el valor $x=4.9651142317961785$ como la raiz obtenida en la evaluación 37.

En cambio, Newton-Raphson alcanza la aproximación $x \approx 4.9651142317442769$ en solo $4$ iteraciones. Necesita evaluar la función y su derivada en cada paso, pero su convergencia es cuadrática ($p=2$). A partir de los errores y la ecuación de orden numerico (adjuntada al final de la respuesta), se obtiene un orden numérico aproximado de $p \approx 1.996$.

Por otro lado, el método Secante obtiene $x \approx 4.9651142317442760$ en aproximadamente $5$ iteraciones, realizando una nueva evaluación de la función por paso y sin necesitar la derivada. Su convergencia es superlineal, con un orden teórico $p \approx 1.618$ y uno hallado de 2.046.

Se puede afirmar que Newton-Raphson y Secante son considerablemente más rápidos que Bisección. Newton ofrece una convergencia más acelerada, mientras que Secante es mas eficiente al evitar el cálculo de la derivada. Sin embargo, aunque Bisección requiere muchas más iteraciones, conserva la convergencia si se parte de un intervalo adecuado, siendo un metodo mas seguro que los otros dos.


In [115]:
#CALCULO DE FORMULA DEL ORDEN NUMERICO PARA NEWTON
import numpy as np

datos   = np.loadtxt("taller_cuerpo_negro/iteraciones_wien.dat")     #columnas: método, iteración, x (llamado del archivo .dat)
metodo  = datos[:, 0].astype(int)         #extraer la primera columna de los datos, y convertir esos valores a enteros (astype(int))
x_newton  = datos[metodo == 2, 2]         #buscar las filas donde la columna de metodo sea igual a 2 (es decir, el metodo newton)
x_secante = datos[metodo == 3, 2]         #buscar las filas donde la columna de metodo sea igual a 3 (es decir, el metodo secante)

alfa = x_newton[-1]                       #raíz de referencia, es el último iterado de Newton

def orden(x_iter):
    e = np.abs(x_iter - alfa)             #calcular el error absoluto de cada interaccion entre x_iter y la raiz
    e = e[e > 1e-14]                      #filtrar los errores que sean mayores a esa tolerancia
    p = np.log(e[2:] / e[1:-1]) / np.log(e[1:-1] / e[:-2])   #realizar la formual de orden de convergencia
    return e, p

e_n, p_newton  = orden(x_newton)
e_s, p_secante = orden(x_secante)

print("Newton  errores:", e_n, " p =", np.round(p_newton, 3))
print("Secante errores:", e_s, " p =", np.round(p_secante, 3))

Newton  errores: [3.35234611e-02 1.98393568e-05 7.11342096e-12]  p = [1.997]
Secante errores: [3.27076722e-02 8.58800608e-04 5.01734919e-07 7.78488385e-12]  p = [2.045 1.487]



# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
2. **Valor inicial.** ¿A qué valor llega Newton desde $x_0 = 0.5$? ¿Por qué es una raíz correcta de la ecuación, pero no la respuesta al problema físico?

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
Al aplicar Newton-Raphson partiendo del valor inicial $x_0=0.5$, el método converge en 6 iteraciones a:
$$x\approx 8.9452237883628477\times10^{-17}$$

El cual se puede decir que es practicamente 0. Cuando analizamos $x=0$, este es una raíz exacta de $F(x)=x-5(1-e^{-x})=0$, ya que $F(0)=0$, es por ello que el método encuentra una solución válida de la ecuación.

No obstante, la variable está definida como $x=\frac{hc}{\lambda k_BT}$, por lo que $x=0$ corresponde al límite de longitud de onda infinita. En ese límite, la radiancia espectral tiende a cero, por lo que no representa el máximo de emisión que se busca en la ley de Wien, su denominador ${\lambda}$ tiende a infinito. Esto sucede ya que, desde $x_0=0.5$, la derivada es $F'(0.5)=1-5e^{-0.5}\approx-2.0326$, es decir, una pendiente negativa que hace que el primer paso de Newton lleve las iteraciones hacia la raíz $x=0$, en lugar de la raíz positiva, la cual es la solución física del problema.


# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --

3. **Parada prematura.** En la Parte C, ¿qué valor da Romberg en $[0, 100]$ y en qué nivel se detiene? Explique, con los puntos que evalúa en sus primeros niveles, por qué el criterio de la diagonal se cumple sin que la integral haya convergido. Proponga una comprobación adicional.

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
En la integración sobre el intervalo $[0,100]$, el método de Romberg obtiene una integral aproximada de:

$$I\approx 1.6072915399699316\times10^{-15}$$

Se detiene prematuramente en el nivel $k=1$, con apenas 3 evaluaciones, y calcula una constante $\sigma\approx1.4034539387964817\times10^{-23}$, muy alejada del valor esperado.

En el nivel $k=0$, Romberg evalúa los extremos $x=0$ y $x=100$, donde la función $f(x)=\frac{x^3}{e^x-1}$ toma valores prácticamente nulos. Por ende, la primera aproximación de trapecio es prácticamente cero, y en el nivel $k=1$, agrega el punto medio $x=50$, donde también es prácticamente nulo. Así, la extrapolación produce $R(1,1)\approx1.607\times10^{-15}$. Como la diferencia entre los elementos de la diagonal es menor que la tolerancia $10^{-8}$, el algoritmo interpreta erróneamente que alcanzó la convergencia, sin haber evaluado la región donde se concentra la mayor contribución de la integral, alrededor de $x\approx2.82$.

Una comprobación podria ser que se realizase un muestreo antes de la función, en una malla gruesa para identificar las regiones donde hay valores significativos, y así asegurar que los puntos de integración cubran dichas regiones.

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --

4. **Eslabón más débil.** En la Parte D, ¿con qué razón bajan los errores de (b) y de (c) al reducir $\Delta\lambda$ a la mitad? ¿Qué limita la precisión de (b)? ¿Por qué el error de (a) no baja en estas mallas?

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
A partir de los datos de `maximo_espectro.dat`, se observa que los tres métodos presentan comportamientos diferentes al reducir $\Delta\lambda$ a la mitad.

- Método (a - np.argmax): El error permanece constante en $2.03949$ nm para todas las mallas, debido a que el método selecciona únicamente el punto de mayor valor entre los nodos disponibles y en todos los casos, devuelve $\lambda=500$ nm, aunque el máximo real está en $\lambda\approx502.03949$ nm. (como muestra la grafica de la parte D).

- Método (b - spline de la derivada central): El error disminuye de $12.9025$ a $3.3932$, luego a $0.8517$ y finalmente a $0.2131$ nm. En cada paso se reduce aproximadamente 4 veces, lo que nos dice que sigue un segundo orden. Ademas, su precisión está limitada por la derivada central numérica, cuyo error también es de segundo orden y se transmite al spline.

- Método (c - spline directo de la función): El error pasa de $1.2021$ a $0.0322$, después a $0.00290$ y finalmente a $0.000313$ nm. Estas reducciones hacen que la convergencia sea más rápida. Sin embargo, esas razones no son constantes ni llegan a $16$ en todas las reducciones, así que con estos cuatro datos no se puede asegurar un orden exacto de convergencia.

Se puede concluir que método (c) ofrece la mayor precisión, mientras que el método (b) está limitado por el error al poseer la derivada mediante diferencias finitas. Por otro lado, el método (a) no mejora porque depende exclusivamente de los puntos discretos de la malla.

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --

5. **Dos máximos.** ¿Por qué el máximo en longitud de onda y el máximo en frecuencia no corresponden al mismo color? ¿Cuál de los dos es "el" color del Sol?

# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
Los máximos en longitud de onda y frecuencia no coinciden porque la relación entre ambas magnitudes es inversa y no lineal, $\nu=\frac{c}{\lambda}$. Al cambiar de una a otra, interviene un factor que modifica la distribución de la energía y desplaza la posición del máximo.

Según los valores de `constante_wien.dat`, para una temperatura solar de $T=5772\ \text{K}$, se obtiene:

- Máximo en longitud de onda: utilizando $x_5=4.9651142317$ y la constante de Wien $b=2.8977719552\times10^{-3}\ \text{m K}$, se obtiene $\lambda_{\max}\approx502.04\ \text{nm}$, correspondiente a una región verde/azulada del espectro visible.

- Máximo en frecuencia: utilizando $x_3=2.8214393721$, se obtiene una longitud de onda equivalente de aproximadamente $\lambda\approx883.5\ \text{nm}$, correspondiente al infrarrojo cercano, fuera del espectro visible (supera los 750 nm)

Por otro lado, ninguno de estos máximos define por sí solo el color del sol. El Sol emite prácticamente todo el espectro visible y la combinación de esas longitudes de onda genera que se visualice una luz blanca. Desde la Tierra se puede observar con un tono ligeramente amarillento debido a la dispersión de luz en la atmósfera.
# -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --

In [133]:
url_imagen = "https://github.com/Jordy-Castillo/FC_Jordy-Castillo-Trabajos/blob/main/Taller08-09/espectro.jpeg?raw=true"
Image(url=url_imagen, width=500)


---
# **PARTE F:**
La celda siguiente **no debe modificarse**. Lee los archivos que escribieron sus programas, recalcula de forma independiente los valores de referencia, y comprueba la forma de cada archivo. También avisa si un archivo parece anterior a su código. El taller exige los cinco PASS.

In [134]:
# ==================== VERIFICACIÓN DEL TALLER 8 Y 9 (no modificar) ====================
import os
import numpy as np
from scipy.interpolate import CubicSpline as _CS

##########################################################################################
#ESTA SECCION SE REALIZA PARA QUE LA VERIFICACIÓN LEYERA DENTRO DE LA CARPETA
#Y NO DEL CONTENT, YA QUE EL TALLER PIDE QUE TODOS LOS ARCHIVOS ESTEN GUARDADOS
#DENTRO DE LA CARPETA "TALLER_CUERPO_NEGRO", POR LO CUAL, SI NO HACEMOS
#ESTE AGREGADO, LA VERIFICACION SIEMPRE DARÁ FAILS PORQUE NUNCA ENCONTRARA LOS
#ARCHIVOS EN EL CONTENT, DEBIDO A QUE ESTAN EN LA CARPETA
#EL RESTO DEL CODIGO ESTA INTACTO

try:
    #Si el directorio actual se borró o no existe, nos rescatamos volviendo a /contenT
    directorio_original = os.getcwd()
except FileNotFoundError:
    os.chdir('/content')
    directorio_original = '/content'

carpeta_taller = os.path.join(directorio_original, "taller_cuerpo_negro")
if os.path.isdir(carpeta_taller):
    os.chdir(carpeta_taller)

##########################################################################################

try:
    H_, C_, KB_ = 6.62607015e-34, 299792458.0, 1.380649e-23

    def raiz_referencia(n):
        x = float(n)
        for _ in range(200):
            x = n*(1 - np.exp(-x))
        return x

    X5, X3 = raiz_referencia(5), raiz_referencia(3)
    B_REF = H_*C_/(KB_*X5)
    I_REF = np.pi**4/15
    S_REF = 2*np.pi*KB_**4/(H_**3*C_**2)*I_REF
    ITER_ESPERADAS = {(5,1):37, (5,2):4, (5,3):5,
                      (3,1):37, (3,2):5, (3,3):6}

    _avisados = set()
    def tabla(dat, fuentes, forma=None):
        for ruta in [dat, *fuentes]:
            if not os.path.isfile(ruta):
                raise ValueError(f"no existe {ruta}; ejecute desde la carpeta del taller")
        antiguos = [f for f in fuentes if os.path.getmtime(dat) < os.path.getmtime(f)]
        if antiguos and dat not in _avisados:
            print(f"[AVISO] {dat} parece anterior a su fuente; vuelva a generar si editó el código.")
            _avisados.add(dat)
        t = np.loadtxt(dat, ndmin=2)
        if forma is not None and t.shape != forma:
            raise ValueError(f"{dat}: forma {t.shape}; se espera {forma}")
        if not np.all(np.isfinite(t)):
            raise ValueError(f"{dat}: contiene NaN o infinitos")
        return t

    def enteros(a, nombre):
        if not np.all(a == np.floor(a)):
            raise ValueError(f"{nombre}: se requieren valores enteros")

    def resultados_wien():
        t = tabla('resultado_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (7,4))
        enteros(t[:,[0,1,3]], 'n, método e iteraciones')
        claves = [tuple(map(int, fila[:2])) for fila in t[:6]]
        if len(set(claves)) != 6 or set(claves) != set(ITER_ESPERADAS):
            raise ValueError('debe aparecer una vez cada combinación de n=5,3 y método=1,2,3')
        if tuple(t[6,:2]) != (5,4):
            raise ValueError('la fila 7 debe identificar n=5, método=4')
        if np.any((t[:,3] < 1) | (t[:,3] > 100)):
            raise ValueError('conteos fuera de 1 a 100')
        return t

    def constantes():
        t = tabla('constante_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (1,3))[0]
        if abs(t[0]-X5)>2e-10 or abs(t[1]-X3)>2e-10:
            raise ValueError('x5 o x3 incorrectos en constante_wien.dat')
        if abs(t[2]-B_REF)/B_REF > 1e-9:
            raise ValueError('constante de Wien incorrecta')
        return t

    resultados = []
    def caso(nombre, funcion):
        try:
            detalle = funcion()
            ok = True
        except Exception as e:
            ok, detalle = False, str(e)
        resultados.append(ok)
        print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ''))

    def c1():
        t = resultados_wien()
        for n, m, raiz, it in t[:6]:
            ref = X5 if n == 5 else X3
            if abs(raiz-ref)>2e-10:
                raise ValueError(f'n={n:g}, método={m:g}: raíz incorrecta')
            esperado = ITER_ESPERADAS[int(n),int(m)]
            if it != esperado:
                raise ValueError(f'n={n:g}, método={m:g}: {it:g} iteraciones; se esperan {esperado}')
        return f'x5={X5:.10f}, x3={X3:.10f}; conteos correctos'

    def historia_referencia(m):
        # Recurrencias independientes para verificar los registros de n=5.
        f = lambda x: x-5*(1-np.exp(-x))
        df = lambda x: 1-5*np.exp(-x)
        a,b,prev = 1.0,10.0,10.0
        x = {1:0., 2:10., 3:9.}[m]
        valores=[]
        for _ in range(ITER_ESPERADAS[5,m]):
            if m==1:
                z=(a+b)/2
                if f(a)*f(z)<=0: b=z
                else: a=z
            elif m==2: z=x-f(x)/df(x)
            else: z=x-f(x)*(x-prev)/(f(x)-f(prev))
            valores.append(z)
            prev,x=x,z
        return np.array(valores)

    def c2():
        t = tabla('iteraciones_wien.dat', ['modulo_ceros.f90','programa_wien.f90'])
        if t.shape[1]!=3: raise ValueError('iteraciones_wien.dat debe tener tres columnas')
        enteros(t[:,:2], 'método y contador')
        if set(t[:,0])!={1,2,3}: raise ValueError('faltan métodos o hay identificadores inválidos')
        for m in range(1,4):
            filas=t[t[:,0]==m]
            ref=historia_referencia(m)
            if len(filas)!=len(ref): raise ValueError(f'método {m}: número de registros incorrecto')
            if not np.array_equal(filas[:,1], np.arange(1,len(ref)+1)):
                raise ValueError(f'método {m}: use contadores consecutivos desde 1, en orden')
            if not np.allclose(filas[:,2],ref,atol=2e-13,rtol=2e-14):
                raise ValueError(f'método {m}: trayectoria incorrecta o pocas cifras en el archivo')
        return 'tres historiales completos'

    def c3():
        x5,x3,b=constantes()
        t=resultados_wien()
        if abs(t[6,2])>1e-8: raise ValueError('Newton desde 0.5 debe llegar a la raíz trivial')
        return f'b={b:.9e} m K; raíz trivial comprobada'

    def c4():
        t=tabla('resultado_sigma.dat', ['modulo_romberg.f90','programa_sigma.f90'], (2,4))
        enteros(t[:,:2], 'nivel y evaluaciones de Romberg')
        for k,ne,I,s in t:
            if k<0 or k>50 or ne<2:
                raise ValueError('nivel o conteo de Romberg inválido')
            if int(ne)!=2**int(k)+1:
                raise ValueError('use nivel k=0 para el trapecio inicial y cuente nodos únicos: 2**k+1')
            factor=2*np.pi*KB_**4/(H_**3*C_**2)
            if not np.isclose(s,factor*I,rtol=1e-8,atol=1e-30):
                raise ValueError('sigma no corresponde a la integral de su fila')
        if abs(t[0,2]-I_REF)>1e-8 or abs(t[0,3]-S_REF)/S_REF>1e-8:
            raise ValueError('la integral transformada o sigma no alcanzan la precisión solicitada')
        prematura=100/6*(4*50**3/np.expm1(50)+100**3/np.expm1(100))
        if t[1,0]!=1 or not np.isclose(t[1,2],prematura,rtol=1e-6,atol=1e-18):
            raise ValueError('fila 2: debe mostrar la parada prematura en k = 1')
        return f'sigma={t[0,3]:.9e} W/m²/K⁴; parada prematura reproducida'

    def c5():
        constantes()
        t=tabla('maximo_espectro.dat', ['maximo_espectro.py','constante_wien.dat'], (4,4))
        if not np.array_equal(t[:,0],[100.,50.,25.,12.5]):
            raise ValueError('use, en este orden, pasos de 100, 50, 25 y 12.5 nm')
        if np.any(t[:,1:]<0): raise ValueError('los errores absolutos no pueden ser negativos')
        ref=B_REF/5772*1e9
        for fila in t:
            dl=fila[0]
            l=np.arange(100.,3000.+dl/2,dl)
            B=1/l**5/np.expm1(H_*C_/(l*1e-9*KB_*5772))
            la=l[np.argmax(B)]
            def cero(s):
                r=s.roots(extrapolate=False)
                r=r[np.isfinite(r)]
                r=r[s(r,1)<0]
                if len(r)==0: raise ValueError('no se encontró candidato a máximo')
                return r[np.argmin(abs(r-la))]
            esperados=[abs(la-ref),
                abs(cero(_CS(l[1:-1],(B[2:]-B[:-2])/(2*dl)))-ref),
                abs(cero(_CS(l,B).derivative())-ref)]
            for nombre,val,esp in zip(['a','b','c'],fila[1:],esperados):
                if abs(val-esp)>1e-4+0.01*esp:
                    raise ValueError(f'paso {dl:g} nm, método {nombre}: error incompatible')
        if not os.path.isfile('espectro_solar.png'):
            raise ValueError('falta espectro_solar.png')
        return 'cuatro mallas correctas'

    caso('1. Raíces e iteraciones de los tres métodos', c1)
    caso('2. Historiales de iteración y convergencia', c2)
    caso('3. Constante de Wien y Newton desde x0=0.5', c3)
    caso('4. Stefan–Boltzmann y parada prematura', c4)
    caso('5. Máximo del espectro solar en cuatro mallas', c5)
    print(f'\nResultado: {sum(resultados)} de {len(resultados)} PASS')

finally:
    #restauramos el directorio de trabajo original
    try:
        os.chdir(directorio_original)
    except:
        os.chdir('/content')

[PASS] 1. Raíces e iteraciones de los tres métodos -> x5=4.9651142317, x3=2.8214393721; conteos correctos
[PASS] 2. Historiales de iteración y convergencia -> tres historiales completos
[PASS] 3. Constante de Wien y Newton desde x0=0.5 -> b=2.897771955e-03 m K; raíz trivial comprobada
[PASS] 4. Stefan–Boltzmann y parada prematura -> sigma=5.670374419e-08 W/m²/K⁴; parada prematura reproducida
[PASS] 5. Máximo del espectro solar en cuatro mallas -> cuatro mallas correctas

Resultado: 5 de 5 PASS


### **CALCULO DEL ERP PARA REPORTE** ###

In [118]:
## constantes fisicas ##
h = 6.62607015e-34       # Constante de Planck [J s]
c = 299792458.0          # Velocidad de la luz en el vacio [m/s]
k = 1.380649e-23         # Constante de Boltzmann [J/K]
pi = np.pi               # Numero pi
T_sol = 5772.0           # Temperatura solar de referencia [K]

#definimos la ecuacion del ERP ##
def ERP(valor_calculado, valor_teorico):
    return abs((valor_calculado - valor_teorico) / valor_teorico) * 100

## definimos las referencias teoricas ##

# x5_teorico es la raiz de x = 5*(1 - e^-x)
x5_teorico = 4.9651142317442769

# b_teorico proviene de la formula de la Ley de Wien
b_teorico = h*c/(k*x5_teorico)

# I_teorico es el valor exacto de la integral pura de Boltzmann
I_teorico = pi**4/15

# sigma_teorico es la constante de Stefan-Boltzmann derivada de la integral analitica
factor_sigma = (2*pi*k**4)/(h**3*c**2)
sigma_teorico = factor_sigma*I_teorico

# lambda_teorica es la longitud de onda maxima exacta dada convertida a nm
lambda_teorico = (b_teorico/T_sol)*1e9


## Cargamos las constantes de Wien ##
wien = np.loadtxt("taller_cuerpo_negro/constante_wien.dat")
x5_calc = wien[0]
x3_calc = wien[1]
b_calc = wien[2]

#Cargamos los resultados de Wien para n=5
## TENEMOS EN CUENTA QUE: Fila 0: Biseccion, Fila 1: Newton, Fila 2: Secante ##
res_wien = np.loadtxt("taller_cuerpo_negro/resultado_wien.dat")
x5_biseccion = res_wien[0, 2]
x5_newton = res_wien[1, 2]
x5_secante = res_wien[2, 2]

#Cargamos las integrales y sigmas de Boltzmann
#Fila 0: Con cambio de variable, Fila 1: Sin cambio de variable x
res_sigma = np.loadtxt("taller_cuerpo_negro/resultado_sigma.dat")
sigma_t_calc = res_sigma[0, 3]
sigma_x_calc = res_sigma[1, 3]

#Cargamos los errores de las mallas (parte d)
#Columnas: 0=paso, 1=error_a, 2=error_b, 3=error_c
res_espectro = np.loadtxt("taller_cuerpo_negro/maximo_espectro.dat")
err_b_calc = res_espectro[3, 2] #Error absoluto del spline b en nm
err_c_calc = res_espectro[3, 3] #Error absoluto del spline c en nm

## calculamos con la funcion definida del ERP)
print(f"ERP Biseccion (x5): {ERP(x5_biseccion, x5_teorico):.6e} %")
print(f"ERP Newton (x5):    {ERP(x5_newton, x5_teorico):.6e} %")
print(f"ERP Secante (x5):   {ERP(x5_secante, x5_teorico):.6e} %")
print(f"ERP Constante b:    {ERP(b_calc, b_teorico):.6e} %")
print(f"ERP Sigma (con cambio t): {ERP(sigma_t_calc, sigma_teorico):.6e} %")
print(f"ERP Sigma (sin cambio x): {ERP(sigma_x_calc, sigma_teorico):.6e} %")

erp_b_spline = (err_b_calc / lambda_teorico) * 100
erp_c_spline = (err_c_calc / lambda_teorico) * 100


print(f"ERP Metodo (b) Spline Derivada:           {erp_b_spline:.16e} %")
print(f"ERP Metodo (c) Spline Directo:             {erp_c_spline:.16e} %")
print(f"Lambda Teorico:                       {lambda_teorico:.16e}")
print(f"Sigma Teorico:                       {sigma_teorico:.16e}")

ERP Biseccion (x5): 1.045325e-09 %
ERP Newton (x5):    0.000000e+00 %
ERP Secante (x5):   1.788838e-14 %
ERP Constante b:    0.000000e+00 %
ERP Sigma (con cambio t): 4.154594e-12 %
ERP Sigma (sin cambio x): 1.000000e+02 %
ERP Metodo (b) Spline Derivada:           4.2451977772146800e-02 %
ERP Metodo (c) Spline Directo:             6.2293821433708415e-05 %
Lambda Teorico:                       5.0203949327532439e+02
Sigma Teorico:                       5.6703744191844307e-08
